# Frozen GEMV confirmation

This updates the benchmark files and compares only default PyTorch and the frozen rows1 / 4-warp GEMV kernel.

Tests K=1536 with N=8960 (original target), 8192 and 9216 (neighboring synthetic widths), 1536 and 256 (attention dimensions). All use M=1 and contiguous stored weights. Seeds 2,3,4 are new; the middle round reverses shape order. Each paired run has 30 alternating samples of 100 graph operations.

This remains a synthetic FP16 graph benchmark, not an end-to-end model measurement. Copy the code cell into the existing session, or pull the updated repository and run the script with --confirm.


In [ ]:
from pathlib import Path
import subprocess

repo = Path('/kaggle/working/fastinfer')
python = '/kaggle/working/fastinfer-t4-py312/bin/python'
assert Path(python).exists(), 'The isolated Python 3.12 environment is required.'
assert (repo / 'benchmarks/matmul_compare.py').exists(), 'Existing fastinfer checkout required.'

files = {'kernels/triton_gemv.py': '"""Single-row FP16 linear projection with contiguous weights W[N, K]."""\n\nimport torch\nimport triton\nimport triton.language as tl\n\n\nGEMV_CONFIGS = {\n    f"gemv_rows{rows}": {"block_rows": rows, "num_warps": 4}\n    for rows in (1, 4, 8)\n}\n\n\n@triton.jit\ndef _gemv(\n    A,\n    W,\n    C,\n    N: tl.constexpr,\n    K: tl.constexpr,\n    ROWS: tl.constexpr,\n    PAD_K: tl.constexpr,\n):\n    rows = tl.program_id(0) * ROWS + tl.arange(0, ROWS)\n    inner = tl.arange(0, PAD_K)\n\n    x = tl.load(A + inner, inner < K, other=0).to(tl.float32)\n    weights = tl.load(\n        W + rows[:, None] * K + inner[None, :],\n        (rows[:, None] < N) & (inner[None, :] < K),\n        other=0,\n    ).to(tl.float32)\n\n    # Parallel sum along each contiguous weight row; no padded output rows.\n    totals = tl.sum(weights * x[None, :], axis=1)\n\n    tl.store(C + rows, totals, rows < N)\n\n\ndef gemv(a, weight, *, block_rows=4, num_warps=4):\n    """Return a @ weight.T. Requires contiguous FP16 a[1,K], weight[N,K]."""\n    if a.ndim != 2 or weight.ndim != 2 or a.shape[0] != 1:\n        raise ValueError("expected a[1,K] and weight[N,K]")\n\n    n, k = weight.shape\n    if a.shape[1] != k or n <= 0 or not 0 < k <= 16384:\n        raise ValueError("matching K in 1..16384 and positive N required")\n\n    if not a.is_cuda or not weight.is_cuda or a.device != weight.device:\n        raise ValueError("inputs must share a CUDA device")\n\n    if a.dtype != torch.float16 or weight.dtype != torch.float16:\n        raise TypeError("FP16 inputs required")\n\n    if not a.is_contiguous() or not weight.is_contiguous():\n        raise ValueError("contiguous input and stored weights required")\n\n    if block_rows not in (1, 4, 8) or num_warps != 4:\n        raise ValueError("supported rows: 1,4,8; warps: 4")\n\n    output = torch.empty((1, n), device=a.device, dtype=a.dtype)\n    _gemv[(triton.cdiv(n, block_rows),)](\n        a,\n        weight,\n        output,\n        n,\n        k,\n        block_rows,\n        triton.next_power_of_2(k),\n        num_warps=num_warps,\n    )\n\n    return output\n', 'tests/test_triton_gemv.py': '"""Check real and masked dimensions for the FP32-reduction GEMV kernel."""\n\nimport unittest\nimport torch\n\ntry:\n    import triton\nexcept ImportError:\n    triton = None\n\n\n@unittest.skipUnless(triton is not None and torch.cuda.is_available(), "CUDA Triton required")\nclass GemvTest(unittest.TestCase):\n    def test_reference_and_edges(self):\n        from fastinfer.kernels.triton_gemv import gemv, GEMV_CONFIGS\n        previous = torch.get_float32_matmul_precision()\n        torch.set_float32_matmul_precision("highest")\n        try:\n            for seed in (0, 1):\n                torch.manual_seed(seed)\n                for n, k in ((1, 1), (13, 65), (256, 1536), (8960, 1536), (17, 8960)):\n                    a = torch.randn((1, k), device="cuda", dtype=torch.float16)\n                    w = torch.randn((n, k), device="cuda", dtype=torch.float16)\n                    expected = a.float() @ w.float().t()\n                    for name, config in GEMV_CONFIGS.items():\n                        with self.subTest(seed=seed, n=n, k=k, config=name):\n                            torch.testing.assert_close(gemv(a, w, **config).float(),\n                                                       expected, rtol=.01, atol=.01)\n        finally:\n            torch.set_float32_matmul_precision(previous)\n\n    def test_cancellation(self):\n        from fastinfer.kernels.triton_gemv import gemv, GEMV_CONFIGS\n        a = torch.ones((1, 4), device="cuda", dtype=torch.float16)\n        w = torch.tensor([[1000, .25, -1000, .25]], device="cuda", dtype=torch.float16)\n        expected = torch.tensor([[.5]], device="cuda", dtype=torch.float16)\n        for config in GEMV_CONFIGS.values():\n            torch.testing.assert_close(gemv(a, w, **config), expected, rtol=0, atol=0)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'benchmarks/matmul_gemv_compare.py': '"""Compare specialized GEMV, tiled matmul and PyTorch on stored linear weights."""\n\nfrom functools import partial\nimport argparse\nimport hashlib\nimport json\nimport runpy\nimport subprocess\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport torch\n\nfrom fastinfer.kernels.triton_matmul import matmul_kernel_resources, triton_matmul\n\n\nfrom fastinfer.kernels.triton_gemv import GEMV_CONFIGS, gemv\n\n\nROOT = Path(__file__).resolve().parents[1]\nCONFIG = dict(block_m=64, block_n=64, block_k=64, num_warps=4)\n\n\ndef run_round(seed, round_index, helpers, *, k=1536, n=8960, confirm=False):\n    torch.manual_seed(seed)\n    a = torch.randn((1, k), device="cuda", dtype=torch.float16)\n    b = torch.randn((k, n), device="cuda", dtype=torch.float16)\n    # Same values, but storage corresponds to contiguous linear weights W[N,K].\n    bt = b.t().contiguous().t()\n    assert torch.equal(b, bt)\n    expected = a.float() @ b.float()\n    weight = bt.t()\n    operations = {"pytorch": lambda x, w: x @ w.t()}\n    if confirm:\n        operations["gemv_rows1"] = partial(gemv, block_rows=1, num_warps=4)\n    else:\n        operations["triton_tiled"] = lambda x, w: triton_matmul(x, w.t(), **CONFIG)\n        operations.update({name: partial(gemv, **config) for name, config in GEMV_CONFIGS.items()})\n    accuracy = {}\n    resources = {} if confirm else {"triton_tiled": matmul_kernel_resources(a, bt, **CONFIG)}\n    inputs = {name: weight for name in operations}\n    for name, op in operations.items():\n        result = op(a, inputs[name])\n        accuracy[name] = helpers["accuracy_against_fp32"](result, expected)\n        if name != "pytorch":\n            torch.testing.assert_close(result.float(), expected, rtol=.01, atol=.01)\n\n    stream = torch.cuda.Stream()\n    stream.wait_stream(torch.cuda.current_stream())\n    with torch.cuda.stream(stream):\n        for _ in range(10):\n            for name, op in operations.items():\n                op(a, inputs[name])\n    torch.cuda.current_stream().wait_stream(stream)\n    torch.cuda.synchronize()\n    graphs, outputs = {}, {}\n    names = list(operations)\n    # Vary graph construction order across repeats too.\n    shift = round_index % len(names)\n    for name in names[shift:] + names[:shift]:\n        graphs[name], outputs[name] = helpers["capture_graph"](\n            operations[name], a, inputs[name], 100)\n    for _ in range(10):\n        for graph in graphs.values():\n            graph.replay()\n    torch.cuda.synchronize()\n    samples = {name: [] for name in names}\n    for repetition in range(30):\n        shift = (repetition + round_index) % len(names)\n        for name in names[shift:] + names[:shift]:\n            start = torch.cuda.Event(enable_timing=True)\n            end = torch.cuda.Event(enable_timing=True)\n            start.record()\n            graphs[name].replay()\n            end.record()\n            end.synchronize()\n            samples[name].append(start.elapsed_time(end) * 1000 / 100)\n    result = {\n        "seed": seed, "round": round_index + 1,\n        "shape": {"m": 1, "k": k, "n": n},\n        "strides": {"a": list(a.stride()), "stored_weight": list(weight.stride())},\n        "accuracy": accuracy, "kernel_resources": resources,\n        "implementations": {name: helpers["summarize"](values)\n                            for name, values in samples.items()},\n    }\n    pt = result["implementations"]["pytorch"]["latency_us"]["median"]\n    for name, timing in result["implementations"].items():\n        us = timing["latency_us"]["median"]\n        errors = accuracy[name]["mismatched_elements"]\n        print(f"K={k}, N={n}, round {round_index + 1}, seed {seed}, {name}: {us:.2f} us | "\n              f"PT/implementation {pt / us:.3f}x | mismatches {errors}", flush=True)\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--confirm", action="store_true",\n                        help="freeze rows1 and test target, neighboring widths and attention shapes")\n    args = parser.parse_args()\n    shapes = [(1536, n) for n in (8960, 8192, 9216, 1536, 256)] if args.confirm else [(1536, 8960)]\n    seeds = (2, 3, 4) if args.confirm else (0, 0, 1)\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA is required")\n    torch.set_float32_matmul_precision("highest")\n    previous = torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = True\n    helpers = runpy.run_path(str(ROOT / "benchmarks/matmul_compare.py"))\n    output = ROOT / "results/raw" / (\n        ("gemv_confirmation_" if args.confirm else "mlp_expand_gemv_")\n        + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")\n    output.parent.mkdir(parents=True, exist_ok=True)\n    report = {\n        "benchmark": "single_row_gemv_confirmation" if args.confirm else "single_row_gemv_candidates",\n        "environment": json.loads(subprocess.check_output(\n            [sys.executable, str(ROOT / "scripts/environment_report.py")],\n            cwd=ROOT, text=True)),\n        "script_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n        "shapes": [{"m": 1, "k": k, "n": n} for k, n in shapes], "dtype": "float16",\n        "seeds": seeds,\n        "tile": None if args.confirm else CONFIG,\n        "gemv_configs": {"gemv_rows1": {"block_rows": 1, "num_warps": 4}} if args.confirm else GEMV_CONFIGS,\n        "device": torch.cuda.current_device(),\n        "allow_fp16_reduced_precision_reduction": True,\n        "timing_mode": "cuda_graph_replay_amortized", "graph_ops": 100,\n        "warmup": 10, "graph_warmup": 10, "repetitions": 30,\n        "measurement_order": "alternating_two_implementations" if args.confirm else "rotating_five_implementations",\n        "correctness": {"reference": "pytorch_float32", "rtol": .01, "atol": .01},\n        "scope": "Synthetic FP16 matmul; transpose/copy preparation excluded from timing. "\n                 "No bias, actual model weights, or end-to-end inference.",\n        "kernel_sha256": hashlib.sha256((ROOT / "kernels/triton_gemv.py").read_bytes()).hexdigest(),\n        "selection_note": "Frozen rows1, new seeds, no configuration selection." if args.confirm else\n                          "Three candidates on target shape; repeat the selected winner separately.",\n        "rounds": [],\n    }\n    try:\n        with torch.inference_mode():\n            for index, seed in enumerate(seeds):\n                # Reverse shape order in the middle round to vary temporal placement.\n                for k, n in (shapes if index % 2 == 0 else shapes[::-1]):\n                    report["rounds"].append(run_round(seed, index, helpers,\n                                                     k=k, n=n, confirm=args.confirm))\n                    output.write_text(json.dumps(report, indent=2) + "\\n")\n                    torch.cuda.empty_cache()\n    finally:\n        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = previous\n    print(f"Saved: {output}")\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, source in files.items():
    (repo / name).write_text(source)
subprocess.run([python, '-m', 'unittest', 'discover', '-s', 'tests',
                '-p', 'test_triton_gemv.py', '-v'], cwd=repo, check=True)
subprocess.run([python, 'benchmarks/matmul_gemv_compare.py', '--confirm'], cwd=repo, check=True)
